To get not normalized permutation importance we sum all computed differences and divide by the number of trees. Normalization is done by dividing not normalized permutation importance by standard error. what does that mean?

This description is talking specifically about **permutation importance in Random Forests**, where each tree gives its own estimate of feature importance. Let's break it down.

## Step 1: Compute importance for each tree

Suppose your random forest has **5 trees**.

For one feature, say **Income**, you measure how much the prediction error increases after shuffling that feature.

You might get:

| Tree | Increase in error after shuffling Income |
| ---- | ---------------------------------------- |
| 1    | 0.12                                     |
| 2    | 0.09                                     |
| 3    | 0.14                                     |
| 4    | 0.10                                     |
| 5    | 0.15                                     |

Each number is the **difference** between

> Error after permutation − Original error

---

## Step 2: Average across trees

The statement

> "sum all computed differences and divide by the number of trees"

means simply taking the average.

[
\frac{0.12+0.09+0.14+0.10+0.15}{5}
=0.12
]

This is called the **unnormalized permutation importance**.

It tells us:

> "On average, shuffling this feature increases prediction error by 0.12."

---

## Step 3: Why normalize?

Not every tree agrees equally.

Imagine another feature:

| Tree | Increase in error |
| ---- | ----------------- |
| 1    | 0.00              |
| 2    | 0.30              |
| 3    | 0.01              |
| 4    | 0.28              |
| 5    | 0.01              |

Average:

[
\frac{0+0.30+0.01+0.28+0.01}{5}=0.12
]

Notice something interesting?

Its average importance is **also 0.12**, but the trees disagree wildly.

Compare that with the first feature:

```
0.12
0.09
0.14
0.10
0.15
```

Every tree says it's important.

The second feature:

```
0.00
0.30
0.01
0.28
0.01
```

Some trees think it's extremely important.
Others think it's useless.

So we should trust the first estimate more.

---

## Step 4: Standard error measures consistency

The **standard error** measures how much the importance values vary across trees.

A small standard error means:

* trees mostly agree
* importance estimate is reliable

A large standard error means:

* trees disagree
* importance estimate is noisy

---

## Step 5: Normalized importance

The formula becomes

[
\text{Normalized Importance}
============================

\frac{\text{Mean Importance}}
{\text{Standard Error}}
]

This is similar to a **signal-to-noise ratio**.

For example,

Feature A

* Mean importance = 0.12
* Standard error = 0.02

Normalized importance

[
0.12/0.02 = 6
]

Feature B

* Mean importance = 0.12
* Standard error = 0.12

Normalized importance

[
0.12/0.12 = 1
]

Although both have the same average importance, Feature A is much more consistently important across trees, so it gets a higher normalized score.

---

### Intuition

Think of asking five doctors whether a treatment works.

**Treatment A**

```
Yes
Yes
Yes
Yes
Yes
```

Everyone agrees.

**Treatment B**

```
Yes
No
Yes
No
Maybe
```

The average opinion might look similar, but confidence is much lower because the opinions are inconsistent.

The **mean importance** tells you **how large the effect is**, while the **standard error** tells you **how consistently that effect appears across the trees**. Dividing by the standard error rewards features that are both important **and** consistently important, while down-weighting features whose importance is highly variable.


# ------------------------------------------------------

# Simple explination of Sklearn random forest feature importance



Yes. I'll explain it from first principles instead of following the article. Once you understand the intuition, you'll be able to read the formulas and tree visualizations in **mlcourse.ai** without getting lost.

---

# The Big Question

Imagine you trained a Random Forest to classify iris flowers.

Now someone asks:

> **"Which feature was the most important?"**

Maybe the features are

| Feature      |
| ------------ |
| Sepal Length |
| Sepal Width  |
| Petal Length |
| Petal Width  |

The model answers

```text
Petal Length : 47%
Petal Width  : 38%
Sepal Length : 10%
Sepal Width  : 5%
```

How did it get these numbers?

That's what this section explains.

---

# Step 1: What makes a split "good"?

Suppose we have this dataset.

| Flower | Species    |
| ------ | ---------- |
| A      | Setosa     |
| B      | Setosa     |
| C      | Versicolor |
| D      | Virginica  |

Initially the node contains

```text
Setosa
Setosa
Versicolor
Virginica
```

This node is mixed.

It contains three classes.

So its impurity is high.

---

Now suppose we split on

```text
Petal Length < 2.5
```

Left child

```text
Setosa
Setosa
```

Right child

```text
Versicolor
Virginica
```

The left child is perfectly pure.

The right child is still mixed.

Overall

the split improved the situation.

---

Suppose instead we split using

```text
Sepal Width < 3
```

Left child

```text
Setosa
Virginica
```

Right child

```text
Setosa
Versicolor
```

Both children are still messy.

Not much improvement.

---

So

Petal Length made a **better split**.

---

# Step 2: How do we measure "better"?

The tree uses an impurity measure.

For classification

* Gini
* Entropy

For regression

* MSE

Suppose before splitting

```text
Gini = 0.66
```

After splitting

Left

```text
0
```

Right

```text
0.25
```

Now the weighted average becomes

```text
0.12
```

The impurity reduction is

```text
0.66 - 0.12 = 0.54
```

That is a **huge improvement.**

---

Suppose another feature gives

```text
0.66 -> 0.61
```

Reduction

```text
0.05
```

Much worse.

---

The tree therefore says

> Petal Length helped much more than Sepal Width.

---

# Step 3: Why is there a weight?

The formula is

[
RI
==

## wI

## w_LI_L

w_RI_R
]

Ignore the symbols.

Let's use numbers.

---

Suppose the root has

100 flowers.

Impurity

```text
0.50
```

Contribution

```text
100 × 0.50 = 50
```

After splitting

Left

60 flowers

Impurity

```text
0.10
```

Contribution

```text
60 × 0.10 = 6
```

Right

40 flowers

Impurity

```text
0.20
```

Contribution

```text
40 × 0.20 = 8
```

Reduction

```text
50 - 6 - 8 = 36
```

That's the impurity reduction.

---

Why multiply by the number of samples?

Because improving a node containing

```text
1000 samples
```

is much more valuable than improving one containing

```text
3 samples.
```

---

# Step 4: Every split earns "points"

Imagine one tree.

```
                Petal Length
                   +40
                  /   \
          Sepal Width  Petal Width
              +5          +25
```

Think of these as points.

Petal Length reduced impurity by

```text
40
```

Petal Width

```text
25
```

Sepal Width

```text
5
```

So far

Feature importance is

| Feature      | Points |
| ------------ | ------ |
| Petal Length | 40     |
| Petal Width  | 25     |
| Sepal Width  | 5      |

---

# Step 5: What if a feature appears many times?

Suppose another branch also uses

```
Petal Length
```

and gains

```text
18
```

Now

| Feature      | Points |
| ------------ | ------ |
| Petal Length | 58     |
| Petal Width  | 25     |
| Sepal Width  | 5      |

We simply add them.

---

This is exactly what

[
\sum RI_i
]

means.

It says

> Add all impurity reductions from nodes where this feature was used.

---

# Step 6: Why normalize?

Current totals

| Feature      | Points |
| ------------ | ------ |
| Petal Length | 58     |
| Petal Width  | 25     |
| Sepal Width  | 5      |

Total

```text
88
```

Now divide everything by

88.

Petal Length

```text
58/88 = 66%
```

Petal Width

```text
25/88 = 28%
```

Sepal Width

```text
5/88 = 6%
```

Now everything sums to

```text
100%
```

These are the feature importances **for one tree**.

---

# Step 7: What about a Random Forest?

Suppose we have

3 trees.

Tree 1

| Feature      | Importance |
| ------------ | ---------- |
| Petal Length | 60         |
| Petal Width  | 30         |
| Sepal Width  | 10         |

Tree 2

| Feature      | Importance |
| ------------ | ---------- |
| Petal Length | 50         |
| Petal Width  | 40         |
| Sepal Width  | 10         |

Tree 3

| Feature      | Importance |
| ------------ | ---------- |
| Petal Length | 55         |
| Petal Width  | 35         |
| Sepal Width  | 10         |

Average

| Feature      | Average |
| ------------ | ------- |
| Petal Length | 55      |
| Petal Width  | 35      |
| Sepal Width  | 10      |

Done.

That is exactly what sklearn returns in

```python
rf.feature_importances_
```

---

# What is sklearn actually doing?

When you run

```python
rf.fit(X, y)
```

sklearn quietly keeps track of

Every split.

For each split it records

* which feature was used
* impurity before split
* impurity after split
* number of samples

Then

```text
Split #1
↓

compute reduction

↓

add reduction to feature's score

↓

repeat for every split

↓

normalize

↓

average across trees
```

No magic.

---

# Why are Petal Length and Petal Width important in Iris?

Think about the data.

Setosa flowers have

```text
Tiny petals
```

Versicolor

```text
Medium petals
```

Virginica

```text
Large petals
```

One split on Petal Length almost separates the species.

So almost every tree starts with

```
Petal Length
```

Because that split removes a huge amount of impurity.

Hence

large importance.

Sepal Width doesn't separate the species nearly as well, so when it's used, it usually gives only a small impurity reduction. Its accumulated score stays low.

---

# The most important insight (the one people often miss)

A feature is **not** important because it appears near the root or because it is used many times.

A feature is important because **every time it is used, it significantly reduces impurity, especially in nodes containing many samples.**

For example:

* Feature A is used **once** at the root and reduces impurity by **120**.
* Feature B is used **10 times** in tiny leaf regions and each split reduces impurity by **2**.

Feature A contributes **120** points.

Feature B contributes **20** points.

So **Feature A is much more important**, even though it was used fewer times.

---

## A mental model to remember forever

Imagine every feature has a **bank account**.

Every time the tree uses a feature to split:

* Compute **how much cleaner** the data became (impurity reduction).
* Deposit that amount into the feature's bank account.

After building one tree:

* Add up each feature's deposits.
* Convert them into percentages (normalize).

After building the whole forest:

* Average those percentages across all trees.

The features with the biggest bank balances are reported as the most important.

If you keep this "bank account" analogy in mind, every formula in the `3. Sklearn Random Forest Feature Importance` section is just the mathematical way of describing deposits, normalization, and averaging.
